In [ ]:
import csv, json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

# ── Config ───────────────────────────────────────────────────────────────────
BENCHMARK_DIR = Path(".")  # relative to repo root
CSV_PATH = BENCHMARK_DIR / "results/vetting_log.csv"
NCOLS = 4
# ─────────────────────────────────────────────────────────────────────────────

from planet_ruler.image import load_image
from planet_ruler.geometry import limb_arc

In [ ]:
with open(CSV_PATH) as f:
    vet_rows = [r for r in csv.DictReader(f) if r.get("best_parameters")]

print(f"{len(vet_rows)} images with saved parameters")
for r in vet_rows:
    print(
        f"  {r['verdict']:4s}  {float(r['fitted_altitude_km']):6.1f} km  {r['image_name']}"
    )

In [ ]:
def load_annotation(annot_path: Path, image_width: int) -> np.ndarray:
    with open(annot_path) as f:
        data = json.load(f)
    points = data.get("limb_points", {}).get("points") or data.get("points", [])
    target = np.full(image_width, np.nan)
    for x, y in points:
        xi = int(round(x))
        if 0 <= xi < image_width:
            target[xi] = y
    return target


VERDICT_COLOR = {"PASS": "#2ecc71", "WARN": "#f39c12", "FAIL": "#e74c3c"}
nrows = (len(vet_rows) + NCOLS - 1) // NCOLS
fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 5, nrows * 4))
axes = axes.flat

for ax, row in zip(axes, vet_rows):
    stem = row["image_name"]
    h_km = float(row["fitted_altitude_km"])
    verdict = row["verdict"]
    params = json.loads(row["best_parameters"])

    image_path = BENCHMARK_DIR / "images" / f"{stem}.jpg"
    annot_name = row["annotation_file"]
    annot_path = BENCHMARK_DIR / "annotations" / annot_name
    if not annot_path.exists():
        annot_path = BENCHMARK_DIR / "images" / annot_name

    img = load_image(str(image_path))
    H, W = img.shape[:2]

    arc = limb_arc(**params)
    target = load_annotation(annot_path, W)

    ax.imshow(img, aspect="auto")
    ax.scatter(np.arange(W), arc, c="yellow", s=2, alpha=0.6, linewidths=0)
    x_ann = np.where(~np.isnan(target))[0]
    ax.scatter(x_ann, target[x_ann], c="red", s=20, zorder=5)

    short = (
        stem.replace("pexels-", "")
        .replace("_exif_cropped", "")
        .replace("_exif", "")[:28]
    )
    ax.set_title(
        f"{short}\nh={h_km:.1f} km  [{verdict}]",
        fontsize=14,
        color="white",
        bbox=dict(
            boxstyle="round,pad=0.2",
            facecolor=VERDICT_COLOR.get(verdict, "grey"),
            alpha=0.85,
        ),
    )
    ax.axis("off")

for ax in list(axes)[len(vet_rows) :]:
    ax.set_visible(False)

fig.suptitle("Vetting — fitted arc (yellow) vs annotations (red)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()